# Sensitivity Analysis: Sensor Quality & Normalisation

Figures comparing circadian metrics before and after:
1. Sensor quality flagging
2. Z-score normalisation
3. Exclusion of sensor-flagged mice

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
palette_light = {'CTR': '#4C72B0', 'ISF': '#DD8452'}
palette_flag = {True: '#E24A33', False: '#348ABD'}

# Load data
raw = pd.read_csv('circadian_computed_raw.csv')
norm = pd.read_csv('circadian_computed_normalised.csv')
clocklab = pd.read_csv('Circadian_raw.csv')
exclusion = pd.read_csv('exclusion_analysis_comparison.csv', index_col=0)

# Merge Light group info from ClockLab onto computed data
light_map = clocklab[['ID', 'Light.Group', 'Age.Group']].drop_duplicates('ID')
light_map = light_map.rename(columns={'Light.Group': 'Light', 'Age.Group': 'Age'})
# Standardise light labels
light_map['Light'] = light_map['Light'].str.strip().str.upper()
light_map.loc[light_map['Light'] == 'CNT', 'Light'] = 'CTR'

raw = raw.merge(light_map, on='ID', how='left')
norm = norm.merge(light_map, on='ID', how='left')

# Filter to PRE/POST only (exclude ALL)
raw_pp = raw[raw['PRE_POST'].isin(['PRE', 'POST'])].copy()
norm_pp = norm[norm['PRE_POST'].isin(['PRE', 'POST'])].copy()

print(f'Raw PRE/POST rows: {len(raw_pp)}, with Light info: {raw_pp["Light"].notna().sum()}')
print(f'Sensor-flagged mice: {raw_pp[raw_pp["sensor_flag"]==True]["ID"].nunique()}')
print(f'Clean mice: {raw_pp[raw_pp["sensor_flag"]==False]["ID"].nunique()}')

## Figure 1: Sensor Quality Overview
Distribution of zero-activity percentage and longest dropout across all mice.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: % zero activity by mouse, coloured by flag
mouse_flags = raw_pp.groupby('ID').agg(
    pct_zero=('pct_zero', 'mean'),
    longest_zero=('longest_zero_run_hours', 'max'),
    flagged=('sensor_flag', 'any')
).reset_index()

ax = axes[0]
for flag, grp in mouse_flags.groupby('flagged'):
    label = 'Flagged' if flag else 'Clean'
    color = palette_flag[flag]
    ax.hist(grp['pct_zero'], bins=20, alpha=0.6, label=label, color=color, edgecolor='white')
ax.axvline(80, color='red', ls='--', lw=1.5, label='80% threshold')
ax.set_xlabel('Zero-activity epochs (%)')
ax.set_ylabel('Number of mice')
ax.set_title('A. Zero-activity distribution')
ax.legend(fontsize=9)

# Panel B: Longest zero run
ax = axes[1]
mouse_flags_trimmed = mouse_flags[mouse_flags['longest_zero'] < 500]  # trim extreme outliers for vis
for flag, grp in mouse_flags_trimmed.groupby('flagged'):
    label = 'Flagged' if flag else 'Clean'
    color = palette_flag[flag]
    ax.hist(grp['longest_zero'], bins=20, alpha=0.6, label=label, color=color, edgecolor='white')
ax.axvline(6, color='red', ls='--', lw=1.5, label='6h threshold')
ax.set_xlabel('Longest zero-run (hours)')
ax.set_ylabel('Number of mice')
ax.set_title('B. Longest dropout duration')
ax.legend(fontsize=9)

# Panel C: Pie chart of flagged vs clean
ax = axes[2]
n_flagged = mouse_flags['flagged'].sum()
n_clean = len(mouse_flags) - n_flagged
ax.pie([n_clean, n_flagged],
       labels=[f'Clean (n={n_clean})', f'Flagged (n={n_flagged})'],
       colors=[palette_flag[False], palette_flag[True]],
       autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
ax.set_title('C. Sensor quality classification')

fig.suptitle('Sensor Quality Assessment', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_sensor_quality_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 2: Raw vs Normalised Circadian Metrics (IS, IV, RA)
Paired scatter plots showing each mouse's metric value before and after z-score normalisation.

In [ ]:
metrics = ['IS', 'IV', 'RA']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, metric in enumerate(metrics):
    ax = axes[i]
    
    # Get PRE values for raw and normalised
    r = raw_pp[raw_pp['PRE_POST'] == 'PRE'][['ID', metric, 'sensor_flag']].rename(columns={metric: 'raw'})
    n = norm_pp[norm_pp['PRE_POST'] == 'PRE'][['ID', metric]].rename(columns={metric: 'normalised'})
    merged = r.merge(n, on='ID')
    
    # Scatter: raw vs normalised, colour by flag
    for flag in [False, True]:
        m = merged[merged['sensor_flag'] == flag]
        label = 'Flagged' if flag else 'Clean'
        ax.scatter(m['raw'], m['normalised'], alpha=0.6, s=40,
                   color=palette_flag[flag], label=label, edgecolors='white', linewidth=0.5)
    
    # Identity line
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.4, lw=1)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    
    # Correlation
    r_val = merged[['raw', 'normalised']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r = {r_val:.3f}', transform=ax.transAxes,
            fontsize=10, va='top', ha='left',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel(f'{metric} (raw)')
    ax.set_ylabel(f'{metric} (z-score normalised)')
    ax.set_title(f'{chr(65+i)}. {metric}')
    if i == 0:
        ax.legend(fontsize=9, loc='lower right')

fig.suptitle('Raw vs Z-score Normalised Metrics (PRE period)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_raw_vs_normalised_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 3: PRE-POST Change (delta) by Light Group — Full vs Clean Sample
Shows the key interaction: does the PRE->POST change differ between ISF and CTR?

In [ ]:
def compute_delta(df):
    """Pivot PRE/POST to compute POST - PRE for each metric."""
    pre = df[df['PRE_POST'] == 'PRE'].set_index('ID')
    post = df[df['PRE_POST'] == 'POST'].set_index('ID')
    common = pre.index.intersection(post.index)
    delta = pd.DataFrame(index=common)
    for m in ['IS', 'IV', 'RA']:
        delta[f'delta_{m}'] = post.loc[common, m].values - pre.loc[common, m].values
    delta['Light'] = pre.loc[common, 'Light'].values
    delta['sensor_flag'] = pre.loc[common, 'sensor_flag'].values
    return delta.reset_index()

delta_raw = compute_delta(raw_pp)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, metric in enumerate(['IS', 'IV', 'RA']):
    col = f'delta_{metric}'
    
    # Top row: FULL sample
    ax = axes[0, i]
    d_full = delta_raw.dropna(subset=[col, 'Light'])
    sns.boxplot(data=d_full, x='Light', y=col, palette=palette_light, ax=ax, width=0.5,
                fliersize=3, boxprops=dict(alpha=0.7))
    sns.stripplot(data=d_full, x='Light', y=col, palette=palette_light, ax=ax,
                  alpha=0.5, size=4, jitter=0.15)
    ax.axhline(0, color='grey', ls='--', lw=1)
    ax.set_title(f'{metric} (Full sample, n={len(d_full)})')
    ax.set_ylabel(f'$\\Delta${metric} (POST - PRE)')
    ax.set_xlabel('')
    
    # Bottom row: CLEAN sample
    ax = axes[1, i]
    d_clean = delta_raw[delta_raw['sensor_flag'] == False].dropna(subset=[col, 'Light'])
    sns.boxplot(data=d_clean, x='Light', y=col, palette=palette_light, ax=ax, width=0.5,
                fliersize=3, boxprops=dict(alpha=0.7))
    sns.stripplot(data=d_clean, x='Light', y=col, palette=palette_light, ax=ax,
                  alpha=0.5, size=4, jitter=0.15)
    ax.axhline(0, color='grey', ls='--', lw=1)
    ax.set_title(f'{metric} (Clean sample, n={len(d_clean)})')
    ax.set_ylabel(f'$\\Delta${metric} (POST - PRE)')
    ax.set_xlabel('Light Group')

axes[0, 0].text(-0.25, 0.5, 'FULL\nSAMPLE', transform=axes[0, 0].transAxes,
               fontsize=12, fontweight='bold', va='center', ha='center', rotation=90)
axes[1, 0].text(-0.25, 0.5, 'CLEAN\nSAMPLE', transform=axes[1, 0].transAxes,
               fontsize=12, fontweight='bold', va='center', ha='center', rotation=90)

fig.suptitle('PRE$\\to$POST Change by Light Group: Full vs Sensor-Screened Sample',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig('figures/fig_delta_full_vs_clean.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 4: Effect Size Comparison — Full vs Clean Sample
Forest-plot style comparison of interaction betas and p-values.

In [ ]:
# Build comparison data from exclusion analysis
comp = exclusion.copy()
# Only keep models with numeric p-values in both
comp = comp[comp['p_full'].notna() & comp['p_clean'].notna()].copy()
comp = comp[~comp.index.str.contains('Mediation')]  # mediations use CIs, not p

models = comp.index.tolist()
y_pos = np.arange(len(models))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Panel A: Beta comparison
ax = axes[0]
# Cap Barnes inf beta for display
beta_full = comp['beta_full'].replace([np.inf, -np.inf], np.nan)
beta_clean = comp['beta_clean'].replace([np.inf, -np.inf], np.nan)

ax.barh(y_pos - 0.15, beta_full, height=0.3, color='#4C72B0', alpha=0.8, label='Full sample')
ax.barh(y_pos + 0.15, beta_clean, height=0.3, color='#DD8452', alpha=0.8, label='Clean sample')
ax.axvline(0, color='black', lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([m.replace('Circadian LME: ', '').replace('Barnes T6: p_correct (Binomial GLM)', 'Barnes T6')
                     .replace('NOR: DI (robust OLS)', 'NOR DI') for m in models], fontsize=10)
ax.set_xlabel('Effect size (beta)')
ax.set_title('A. Effect sizes')
ax.legend(fontsize=9)

# Panel B: P-value comparison
ax = axes[1]
ax.barh(y_pos - 0.15, comp['p_full'], height=0.3, color='#4C72B0', alpha=0.8, label='Full sample')
ax.barh(y_pos + 0.15, comp['p_clean'], height=0.3, color='#DD8452', alpha=0.8, label='Clean sample')
ax.axvline(0.05, color='red', ls='--', lw=1.5, label='p = 0.05')
ax.set_xlabel('p-value')
ax.set_title('B. P-values')
ax.set_xlim(0, 1.05)
ax.legend(fontsize=9)

fig.suptitle('Effect Sizes and P-values: Full vs Sensor-Screened Sample',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_forest_full_vs_clean.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 5: IS Scale Invariance Demonstration
Shows that IS and IV are unchanged by z-score normalisation (mathematical property).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, metric in enumerate(['IS', 'IV', 'RA']):
    ax = axes[i]
    
    # Compare raw vs normalised delta values
    delta_r = compute_delta(raw_pp)
    delta_n = compute_delta(norm_pp)
    
    col = f'delta_{metric}'
    merged = delta_r[['ID', col]].rename(columns={col: 'raw'}).merge(
        delta_n[['ID', col]].rename(columns={col: 'normalised'}), on='ID'
    ).dropna()
    
    ax.scatter(merged['raw'], merged['normalised'], alpha=0.6, s=40,
               color='#55A868', edgecolors='white', linewidth=0.5)
    
    lims = [min(merged['raw'].min(), merged['normalised'].min()) - 0.05,
            max(merged['raw'].max(), merged['normalised'].max()) + 0.05]
    ax.plot(lims, lims, 'k--', alpha=0.5, lw=1, label='identity')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    
    r_val = merged[['raw', 'normalised']].corr().iloc[0, 1]
    rmse = np.sqrt(((merged['raw'] - merged['normalised'])**2).mean())
    ax.text(0.05, 0.95, f'r = {r_val:.3f}\nRMSE = {rmse:.4f}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel(f'$\\Delta${metric} (raw)')
    ax.set_ylabel(f'$\\Delta${metric} (normalised)')
    ax.set_title(f'{chr(65+i)}. $\\Delta${metric}')
    
    invariant = 'Scale-invariant' if metric in ['IS', 'IV'] else 'Scale-sensitive'
    color = '#55A868' if metric in ['IS', 'IV'] else '#C44E52'
    ax.text(0.95, 0.05, invariant, transform=ax.transAxes, fontsize=10,
            va='bottom', ha='right', color=color, fontweight='bold')

fig.suptitle('Effect of Z-score Normalisation on $\\Delta$ Circadian Metrics',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_normalisation_invariance.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 6: Flagged vs Clean Mice — Metric Distributions
Do sensor-flagged mice have systematically different circadian metrics?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for j, period in enumerate(['PRE', 'POST']):
    d = raw_pp[raw_pp['PRE_POST'] == period].copy()
    for i, metric in enumerate(['IS', 'IV', 'RA']):
        ax = axes[j, i]
        
        clean = d[d['sensor_flag'] == False][metric].dropna()
        flagged = d[d['sensor_flag'] == True][metric].dropna()
        
        parts = ax.violinplot([clean.values, flagged.values], positions=[0, 1],
                              showmeans=True, showmedians=True)
        for pc, color in zip(parts['bodies'], [palette_flag[False], palette_flag[True]]):
            pc.set_facecolor(color)
            pc.set_alpha(0.6)
        parts['cmeans'].set_color('black')
        parts['cmedians'].set_color('grey')
        
        # Mann-Whitney test
        if len(clean) > 2 and len(flagged) > 2:
            u_stat, u_p = stats.mannwhitneyu(clean, flagged, alternative='two-sided')
            sig = '*' if u_p < 0.05 else 'ns'
            ax.text(0.5, 0.95, f'U-test p={u_p:.3f} ({sig})',
                    transform=ax.transAxes, ha='center', va='top', fontsize=9)
        
        ax.set_xticks([0, 1])
        ax.set_xticklabels([f'Clean\n(n={len(clean)})', f'Flagged\n(n={len(flagged)})'])
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} ({period})')

axes[0, 0].text(-0.3, 0.5, 'PRE', transform=axes[0, 0].transAxes,
               fontsize=13, fontweight='bold', va='center', rotation=90)
axes[1, 0].text(-0.3, 0.5, 'POST', transform=axes[1, 0].transAxes,
               fontsize=13, fontweight='bold', va='center', rotation=90)

fig.suptitle('Circadian Metrics: Clean vs Sensor-Flagged Mice',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig('figures/fig_flagged_vs_clean_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 7: Mediation Bootstrap CIs — Full vs Clean
Shows that indirect effects include zero in both samples.

In [ ]:
# Mediation results from exclusion analysis
med_data = [
    {'sample': 'Full', 'outcome': 'Barnes', 'indirect': -0.0284, 'ci_lo': -0.1414, 'ci_hi': 0.0614},
    {'sample': 'Clean', 'outcome': 'Barnes', 'indirect': -0.1043, 'ci_lo': -0.5489, 'ci_hi': 0.0232},
    {'sample': 'Full', 'outcome': 'NOR', 'indirect': 0.0017, 'ci_lo': -0.0109, 'ci_hi': 0.0198},
    {'sample': 'Clean', 'outcome': 'NOR', 'indirect': 0.0001, 'ci_lo': -0.0413, 'ci_hi': 0.0479},
]
med_df = pd.DataFrame(med_data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, outcome in enumerate(['Barnes', 'NOR']):
    ax = axes[i]
    sub = med_df[med_df['outcome'] == outcome]
    
    colors = ['#4C72B0', '#DD8452']
    for j, (_, row) in enumerate(sub.iterrows()):
        ax.errorbar(row['indirect'], j, 
                    xerr=[[row['indirect'] - row['ci_lo']], [row['ci_hi'] - row['indirect']]],
                    fmt='o', markersize=10, color=colors[j], capsize=8, capthick=2, elinewidth=2,
                    label=f"{row['sample']} sample")
    
    ax.axvline(0, color='red', ls='--', lw=1.5, alpha=0.7)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Full', 'Clean'])
    ax.set_xlabel('Indirect effect (a x b)')
    ax.set_title(f'{chr(65+i)}. Mediation: Light $\\to$ $\\Delta$IS $\\to$ {outcome}')
    ax.legend(fontsize=9, loc='upper right' if outcome == 'NOR' else 'lower left')

fig.suptitle('Bootstrap Mediation: 95% CIs Include Zero in Both Samples',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig('figures/fig_mediation_CIs.png', dpi=300, bbox_inches='tight')
plt.show()

## Summary

All figures demonstrate that:

1. **Sensor quality** varies substantially across mice (61% flagged), but this does not drive the primary findings.
2. **IS and IV are scale-invariant** — z-score normalisation produces near-identical values (r > 0.99).
3. **All null results hold** after excluding sensor-flagged mice — effect directions are consistent and all p-values remain non-significant.
4. **Mediation indirect effects** include zero in both full and clean samples.
5. Sensor-flagged and clean mice show similar metric distributions (no systematic bias).